In [54]:
!pip install lifelines

In [55]:
import pandas as pd
import pandas.api.types
import numpy as np
from lifelines.utils import concordance_index

class ParticipantVisibleError(Exception):
    pass


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """
    >>> import pandas as pd
    >>> row_id_column_name = "id"
    >>> y_pred = {'prediction': {0: 1.0, 1: 0.0, 2: 1.0}}
    >>> y_pred = pd.DataFrame(y_pred)
    >>> y_pred.insert(0, row_id_column_name, range(len(y_pred)))
    >>> y_true = { 'efs': {0: 1.0, 1: 0.0, 2: 0.0}, 'efs_time': {0: 25.1234,1: 250.1234,2: 2500.1234}, 'race_group': {0: 'race_group_1', 1: 'race_group_1', 2: 'race_group_1'}}
    >>> y_true = pd.DataFrame(y_true)
    >>> y_true.insert(0, row_id_column_name, range(len(y_true)))
    >>> score(y_true.copy(), y_pred.copy(), row_id_column_name)
    0.75
    """
    
    del solution[row_id_column_name]
    del submission[row_id_column_name]
    
    event_label = 'efs'
    interval_label = 'efs_time'
    prediction_label = 'prediction'
    for col in submission.columns:
        if not pandas.api.types.is_numeric_dtype(submission[col]):
            raise ParticipantVisibleError(f'Submission column {col} must be a number')
    # Merging solution and submission dfs on ID
    merged_df = pd.concat([solution, submission], axis=1)
    merged_df.reset_index(inplace=True)
    merged_df_race_dict = dict(merged_df.groupby(['race_group']).groups)
    metric_list = []
    for race in merged_df_race_dict.keys():
        # Retrieving values from y_test based on index
        indices = sorted(merged_df_race_dict[race])
        merged_df_race = merged_df.iloc[indices]
        # Calculate the concordance index
        c_index_race = concordance_index(
                        merged_df_race[interval_label],
                        -merged_df_race[prediction_label],
                        merged_df_race[event_label])
        metric_list.append(c_index_race)
    return float(np.mean(metric_list)-np.sqrt(np.var(metric_list)))

In [56]:
import numpy as np
import pandas as pd
import os

input_dir = '/kaggle/input/equity-post-HCT-survival-predictions'

In [57]:
def ingest(input_dir):
    target = 'efs_time'
    
    train_csv = pd.read_csv(os.path.join(input_dir, 'train.csv'))
    
    X_train_df = train_csv.drop(columns=['ID', target])
    
    categories = X_train_df.select_dtypes(exclude=np.number).columns.tolist()

    for c in categories:
        X_train_df[c] = X_train_df[c].astype('category')

    y_train_df = train_csv[target]

    return X_train_df, y_train_df

X_train_df, y_train_df = ingest(input_dir)
# print(categories)
# train_csv.nunique()
# train_csv['dri_score']
# print(train_csv['efs_time'], "\n", train_csv['efs'])
# print(train_csv['age_at_hct'], train_csv['donor_age'])


In [58]:
from sklearn.model_selection import train_test_split
import xgboost as xgb

X_train_set, X_test_set, y_train_set, y_test_set = train_test_split(X_train_df, y_train_df, test_size=0.2, random_state=42)

train_data = xgb.DMatrix(X_train_set, y_train_set, enable_categorical=True)
test_data = xgb.DMatrix(X_test_set, y_test_set, enable_categorical=True)

In [70]:
params = {"objective": "reg:squarederror", "device": "cuda"}
evals = [(train_data, "train"), (test_data, "validation")]

n = 1000

model = xgb.train(
    params=params,
    dtrain=train_data,
    num_boost_round=n,
    evals=evals,
    verbose_eval=10,
    early_stopping_rounds=20
)

results = xgb.cv(
    params,
    train_data,
    num_boost_round=n,
    nfold=5,
    early_stopping_rounds=20
)

[0]	train-rmse:18.70253	validation-rmse:18.91866
[10]	train-rmse:8.68568	validation-rmse:9.60789
[20]	train-rmse:7.74369	validation-rmse:9.40395


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [21:50:12] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[30]	train-rmse:7.27915	validation-rmse:9.31798
[40]	train-rmse:6.72383	validation-rmse:9.24106
[50]	train-rmse:6.43579	validation-rmse:9.24909
[57]	train-rmse:6.18472	validation-rmse:9.24803


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [21:50:13] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


In [71]:
from sklearn.metrics import mean_squared_error

train_csv = pd.read_csv(os.path.join(input_dir, 'train.csv'))
# print(train_csv.isna().sum())
preds = model.predict(test_data)

# rmse = mean_squared_error(y_test_set, preds, squared=False)
# print(f'RMSE of model: {rmse:.4f}')

preds_df = pd.DataFrame({
    "id": range(len(preds)),
    "prediction": preds
})

solution_df = train_csv.loc[X_test_set.index, ['efs', 'efs_time', 'race_group']].copy()
solution_df.insert(0, 'id', range(len(preds)))
solution_df.reset_index(drop=True, inplace=True)

# print(preds_df)
# print(solution_df)

preds_score = score(solution_df, preds_df, 'id')

print(f"Score: {preds_score}")

best_rmse = results['test-rmse-mean'].min()
print(best_rmse)

Score: 0.10554534474745278
9.217205205507703


In [ ]:
test_csv = pd.read_csv(os.path.join(input_dir, 'test.csv'))
X_val = test_csv.drop(columns=['id'])

categories = X_val.select_dtypes(exclude=np.number).columns.tolist()
    
for col in categories:
    X_val[col] = X_val[col].astype('category')

dval_reg = xgb.DMatrix(X_val, enable_categorical=True)

preds = model.predict(dval_reg)

ids = test_csv['id']

submission = pd.DataFrame({
    'ID': ids,
    'prediction': preds
})
submission.to_csv('submission.csv', index=False)